In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
# import threading
mt5.initialize()


True

In [2]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("EURUSD", 0.02, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

38.78

In [32]:
def get_values(symbol):
    timezone = pytz.timezone("Etc/UTC")
    x = datetime.now()
    utc_from = datetime(2021, 7, 1, tzinfo=timezone)
    utc_to = datetime(x.year, x.month+1, x.day, tzinfo=timezone)
    # utc_to = datetime(x.year, x.month+1, 1, tzinfo=timezone)
    rates = mt5.copy_rates_range(symbol, mt5.TIMEFRAME_M30, utc_from, utc_to)

    rates_frame = pd.DataFrame(rates)
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume'], axis=1)
    # convert time in seconds into the datetime format
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
#     rates_frame['rsi'] = pta.rsi(rates_frame['close'], length = 14)
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=100).mean()
#     rates_frame['smaH'] = rates_frame['high'].rolling(window=20).mean()
#     rates_frame['smaL']= rates_frame['low'].rolling(window=20).mean()
    rates_frame = rates_frame.drop(['high', 'low'], axis=1)
    return rates_frame

In [ ]:
def get_rsi(close, lookback):
    ret = close.diff()
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
    rsi_df = rsi_df.dropna()
    return rsi_df[3:]

# ibm['rsi_14'] = get_rsi(ibm['close'], 14)
# ibm = ibm.dropna()


In [ ]:
# symbol = "GBPUSD"
# a= get_values(symbol)
# a['rsi'] = get_rsi(a['close'], 14)
a = a.dropna()

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
    if str(a.iloc[i].rsi) != 'nan':
        if round(a.iloc[i].rsi, 2) > 62.0 and int(round(a.iloc[i-1].rsi, 2)*100) in range(int(59.0*100), int(63.5*100)) \
        and a.iloc[i].rsi > a.iloc[i-1].rsi and a.iloc[i-2].rsi > a.iloc[i-1].rsi \
        and a.iloc[i-2].rsi > 67.0 and check == 0 :
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print(round(a.iloc[i].rsi,2))
            
            print("*"*20)
            check = 1  

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---", round(a.iloc[i].rsi, 2))
            
            if pp < -3.0:
                profit.append(pp)
                check = 0
            elif round(a.iloc[i].rsi, 2) <= 59.0:
                profit.append(pp)
                check = 0

In [ ]:
sum(profit)

In [ ]:
n = 0
p = 0
tn = 0
tp = 0
for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}")      

In [ ]:
counterr = 0
peck = 0
for i in profit:
    if i < 0.0:
        counterr = counterr + i
    else:
        peck = peck + i
print(counterr)
print(peck)

In [ ]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
    if str(a.iloc[i].sma) != 'nan':
        
        if a.iloc[i].close > a.iloc[i].smaH and a.iloc[i].open > a.iloc[i].smaH \
            and a.iloc[i-1].close > a.iloc[i-1].smaH and a.iloc[i-1].open > a.iloc[i-1].smaH \
            and up == 0:
            up = 1
        elif up == 1 and a.iloc[i].close < a.iloc[i].smaL and a.iloc[i].open < a.iloc[i].smaL and check == 0:
            #if candle has crossed smaL
            if a.iloc[i].close < a.iloc[i].open \
            and int(a.iloc[i].sma*mul) not in range(int(a.iloc[i].smaL*mul), int(a.iloc[i].smaH*mul)) \
            and a.iloc[i].close < a.iloc[i].sma and a.iloc[i].open < a.iloc[i].sma: 
                #checks if next candle is not in reverse of sell or up candle
                buy_price = a.iloc[i].close
                print("#"*20)
                print(a.iloc[i].name)
                print("*"*20)
                check = 1  
                peck = 0

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_SELL)
            print(pp,"---",a.iloc[i].name)
#             profit.append(pp)
#             check = 0
#             up = 0
                
#             if pp < -2.0 and peck == 0:
#                 print(a.iloc[i].name)
#                 profit.append(pp)
#                 check = 0
#                 up = 0
#                 peck = 1
            if a.iloc[i].close > a.iloc[i].smaL:
#                 if peck == 1:
                    print(a.iloc[i].name)
                    profit.append(pp)
                    check = 0
                    up = 0
#                 else:
#                     print(a.iloc[i].name)
#                     profit.append(2*pp)
#                     check = 0
#                     up = 0
            
            
#             if 

            
            


In [33]:
symbol = "EURUSD"
a= get_values(symbol)
a

,open,close,sma
time,,,
2021-07-01 00:00:00,1.18559,1.18555,NaN
2021-07-01 00:30:00,1.18555,1.18581,NaN
2021-07-01 01:00:00,1.18572,1.18574,NaN
2021-07-01 01:30:00,1.18574,1.18575,NaN
2021-07-01 02:00:00,1.18575,1.18581,NaN
...,...,...,...
2021-08-12 08:30:00,1.17435,1.17415,1.172777
2021-08-12 09:00:00,1.17416,1.17385,1.172778
2021-08-12 09:30:00,1.17385,1.17357,1.172775


In [34]:
B = []
check = 0
profit = []
index = []
indexB = []
counter = 0
peck = 0
p= []
checks = 0
counterr = 0
profits = []
up = 0
mul = 100000

for i in range(7, len(a)):
#     if str(a.iloc[i].sma) != 'nan':
        hd = a.iloc[i].close - a.iloc[i].open
        ld = a.iloc[i].open - a.iloc[i].close
        base = 1.17610 - 1.17600
        
#         if and a.iloc[i].open < a.iloc[i].smaL and a.iloc[i].close < a.iloc[i].smaL \
        if a.iloc[i].open < a.iloc[i].close \
        and a.iloc[i-1].open > a.iloc[i-1].close \
        and a.iloc[i-2].open > a.iloc[i-2].close \
        and check == 0 :
#         and hd > base:           
            buy_price = a.iloc[i].close
            print("#"*20)
            print(a.iloc[i].name)
            print("*"*20)
            check = 1  
            peck = 0
            if a.iloc[i].open > a.iloc[i].sma:
                index.append("above")
            else:
                index.append("below")

        elif check == 1:
            sell_price = a.iloc[i].close
            pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
            print(pp,"---",a.iloc[i].name)
            
            check = 0
            if pp < 0.0:
                sell_price = a.iloc[i+1].close
                pp = price_action(symbol, 0.02, buy_price, sell_price,mt5.ORDER_TYPE_BUY)
                
                profit.append(pp)
            else:
                profit.append(pp)

####################
2021-07-01 04:00:00
********************
0.3 --- 2021-07-01 04:30:00
####################
2021-07-01 11:30:00
********************
1.3 --- 2021-07-01 12:00:00
####################
2021-07-01 20:00:00
********************
-0.66 --- 2021-07-01 20:30:00
####################
2021-07-01 22:00:00
********************
0.08 --- 2021-07-01 22:30:00
####################
2021-07-02 04:00:00
********************
-0.4 --- 2021-07-02 04:30:00
####################
2021-07-02 06:00:00
********************
-0.12 --- 2021-07-02 06:30:00
####################
2021-07-02 09:30:00
********************
-2.14 --- 2021-07-02 10:00:00
####################
2021-07-02 11:00:00
********************
-0.62 --- 2021-07-02 11:30:00
####################
2021-07-02 13:00:00
********************
0.02 --- 2021-07-02 13:30:00
####################
2021-07-02 15:00:00
********************
5.04 --- 2021-07-02 15:30:00
####################
2021-07-02 20:00:00
********************
2.38 --- 2021-07-02 20:30:

####################
2021-07-27 10:30:00
********************
-1.8199999999999998 --- 2021-07-27 11:00:00
####################
2021-07-27 17:30:00
********************
2.04 --- 2021-07-27 18:00:00
####################
2021-07-27 19:30:00
********************
0.08 --- 2021-07-27 20:00:00
####################
2021-07-28 00:00:00
********************
0.24 --- 2021-07-28 00:30:00
####################
2021-07-28 04:00:00
********************
0.2 --- 2021-07-28 04:30:00
####################
2021-07-28 08:30:00
********************
-0.48 --- 2021-07-28 09:00:00
####################
2021-07-28 10:00:00
********************
-0.94 --- 2021-07-28 10:30:00
####################
2021-07-28 11:30:00
********************
-1.3 --- 2021-07-28 12:00:00
####################
2021-07-28 14:30:00
********************
-2.02 --- 2021-07-28 15:00:00
####################
2021-07-28 17:00:00
********************
1.6600000000000001 --- 2021-07-28 17:30:00
####################
2021-07-28 21:00:00
******************

In [35]:
n = 0
p = 0
tn = 0
tp = 0
for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}")      

Total negative sm -->-88.47999999999998
Total negative -->73
Total positive sm -->93.6
Total positive -->106


In [47]:
n = 0
p = 0
tn = 0
tp = 0
for i in range(len(profit)):
    if profit[i] < 0.0:
        if index[i] == "above":
            n = n + 1
        else:
            p = p+1
    elif profit[i] >= 0.0:
        if index[i] == "above":
            tn = tn + 1
        else:
            tp = tp+1

In [48]:
print(n, p)
print(tn, tp)

19 54
36 70


In [49]:
19+54+36+70

179

In [46]:
106+73

179